# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.84230735 -0.7930839  -0.21782058 -0.93595877  0.89050902]
 [ 0.49735451  0.17025149 -0.78131084  0.59384     0.98360409]
 [ 0.30517429 -0.29560722  0.20366227 -0.37783904  0.2622089 ]
 [-0.23778792  0.78272781  0.19192816 -0.86158396  0.9549153 ]
 [ 0.92053309  0.85708726  0.3842463  -0.78556959 -0.97587206]
 [ 0.27191088  0.92216299  0.09837304  0.62862414 -0.03874212]
 [ 0.32791887  0.99166268  0.68732183 -0.36422688 -0.43955119]
 [-0.72643427  0.08223011 -0.46860813 -0.20854568 -0.93938571]
 [ 0.55890178  0.08606741 -0.31237511 -0.54918055  0.03730473]
 [ 0.58725714 -0.49648784 -0.58388311  0.41254251 -0.59144787]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a1', 'a2', 'a1', 'a2', 'a2', 'a1', 'a2', 'a2', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:39,  1.18s/it]

SVI:   3%|▎         | 1/34 [00:01<00:39,  1.18s/it, loss=3162.4453]

SVI:   6%|▌         | 2/34 [00:01<00:37,  1.18s/it, loss=2155.3005]

SVI:   9%|▉         | 3/34 [00:01<00:36,  1.18s/it, loss=3232.3057]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.06it/s, loss=2967.8376]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.06it/s, loss=1973.3033]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.06it/s, loss=3109.9373]

SVI: 100%|██████████| 34/34 [00:01<00:00, 20.50it/s, loss=3109.9373]

SVI: 100%|██████████| 34/34 [00:01<00:00, 20.50it/s, loss=1162.3602]